# Component report

Copyright 2023 Google LLC. Licensed under Apache-2.0. Adapted from Qualtran v0.7.0 dev_tools/bloq-report-card.ipynb at bcb17bb25360ed68ce3bc084645db3e929799389.

The existing WordSwap example supplies a real register Program and physical lowering to the five shared checks. The report uses native package-owner labels. Empty owner groups remain visible. The original overlapping Other and factoring-summary query behavior is preserved and recorded separately.


In [ ]:
from collections import Counter

from examples.algorithms.register_component_tutorial import WordSwap
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir import CircuitIR
from flagquantum.core.ir.program import ProgramRecord
from flagquantum.core.ir.program.register_types import unsigned_register_type
from flagquantum.testing.component_examples import ComponentExample
from flagquantum.testing.example_checks import (
    CheckKind,
    CheckResult,
    RegisterExample,
    check_example,
)
from tools.development.component_report import (
    get_report,
    show_report,
    summarize_results,
)


def make_program() -> ProgramRecord:
    return WordSwap(2).decompose(implemented=True)


def lower_program(record: ProgramRecord) -> CircuitIR:
    return bind_register_program(record, ((0, 1), (2, 3)), entry="Swap").circuit


example: ComponentExample[object] = ComponentExample(
    make_program, "two_bit_word_swap", ProgramRecord
)
declared_counts = {"cx": 6}
observed_counts = Counter(
    gate.name for gate in lower_program(make_program()).instructions
)


def check_component(example: ComponentExample[object], kind: CheckKind) -> CheckResult:
    dtype = unsigned_register_type(2)
    fixture = RegisterExample(
        example.make, lower_program, lambda: declared_counts, ((dtype, dtype),)
    )
    return check_example(fixture, kind)


report_card = get_report(
    [ProgramRecord],
    [example],
    package_prefix="flagquantum.",
    check=check_component,
    is_component=lambda cls: cls is ProgramRecord,
)
print("Measured primitive counts", dict(observed_counts))
print("Declared primitive counts", declared_counts)


### All

### `core.ir.program`

In [ ]:
show_report(report_card.query('package == "core.ir.program"'))


In [ ]:
summarize_results(report_card.query('package == "core.ir.program"'))


### `compiler.registers`

In [ ]:
show_report(report_card.query('package == "compiler.registers"'))


In [ ]:
summarize_results(report_card.query('package == "compiler.registers"'))


### `algorithms.chemistry`

In [ ]:
show_report(report_card.query('package.str.startswith("algorithms.chemistry")'))


In [ ]:
summarize_results(report_card.query('package.str.startswith("algorithms.chemistry")'))


### `algorithms.primitives`

In [ ]:
show_report(report_card.query('package == "algorithms.primitives"'))


In [ ]:
summarize_results(report_card.query('package.str.startswith("algorithms.primitives")'))


### Other

In [ ]:
show_report(
    report_card.query(
        "package not in "
        '["core.ir.program",'
        ' "compiler.registers", '
        ' "algorithms.primitives"]'
    ).query('not package.str.startswith("flagquantum.algorithms.chemistry")')
)


### Summary of All

In [ ]:
summarize_results(report_card)
